## LSTM

In [1]:
import pandas as pd
import numpy as np
import yfinance as yf
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

### 1. 종목, 기간 설정

In [2]:
# 종목 설정
symbols = ["INTC"]

all_data = []
start = "2021-01-01"
end = "2024-09-30"

### 2. 피처 엔지니어링 : MACD, 볼린저밴드, RSI

In [3]:
# MACD 계산 함수
def add_macd(df, short_window=12, long_window=26, signal_window=9):
    df['EMA12'] = df['Close'].ewm(span=short_window, adjust=False).mean()
    df['EMA26'] = df['Close'].ewm(span=long_window, adjust=False).mean()
    df['MACD'] = df['EMA12'] - df['EMA26']
    df['Signal_Line'] = df['MACD'].ewm(span=signal_window, adjust=False).mean()
    df['MACD_Hist'] = df['MACD'] - df['Signal_Line']
    return df

# RSI 지표 추가 함수
def add_rsi(df, window=14):
    delta = df['Close'].diff(1)
    gain = np.where(delta > 0, delta, 0).flatten()  # 1차원 배열로 변환
    loss = np.where(delta < 0, -delta, 0).flatten()  # 1차원 배열로 변환

    avg_gain = pd.Series(gain).rolling(window=window, min_periods=1).mean()
    avg_loss = pd.Series(loss).rolling(window=window, min_periods=1).mean()
    rs = avg_gain / avg_loss
    df['RSI'] = 100 - (100 / (1 + rs))
    return df

### 3. 데이터 전처리

In [4]:
# 결측치가 있는 각 열에 대해 전날과 다음날의 평균으로 채우기
def fill_missing_with_neighbors(df, columns):
    for col in columns:
        df[col] = df[col].fillna((df[col].shift(1) + df[col].shift(-1)) / 2)
    return df

In [5]:
# 데이터 가져오기, 지표 추가, 스케일링
for symbol in symbols:
    df = yf.download(symbol, start, end)
    df = add_macd(df)
    df = add_rsi(df)
    
    # 타겟 생성 (다음 날의 주가 상승 여부)
    df['Target'] = (df['Close'].shift(-1) > df['Close']).astype(int)

    columns_to_fill = ['Open', 'High', 'Low', 'Close', 'Volume', 'MACD', 'Signal_Line', 'MACD_Hist', 'RSI']
    df = fill_missing_with_neighbors(df, columns_to_fill)

    # 필요한 컬럼들 스케일링
    scale_cols = ['Open', 'High', 'Low', 'Close', 'Volume', 'MACD', 'Signal_Line', 'MACD_Hist', 'RSI']
    scaler = MinMaxScaler()
    scaled = scaler.fit_transform(df[scale_cols])
    scaled_df = pd.DataFrame(scaled, columns=scale_cols, index=df.index)
    scaled_df['Target'] = df['Target'].values
    all_data.append(scaled_df)

data = pd.concat(all_data, axis=0, ignore_index=False)
data.index = pd.to_datetime(data.index)
data = data.sort_index()

[*********************100%***********************]  1 of 1 completed
C:\Users\조영서\AppData\Roaming\Python\Python311\site-packages\sklearn\utils\_array_api.py:695: RuntimeWarning: All-NaN slice encountered
  return xp.asarray(numpy.nanmin(X, axis=axis))
C:\Users\조영서\AppData\Roaming\Python\Python311\site-packages\sklearn\utils\_array_api.py:712: RuntimeWarning: All-NaN slice encountered
  return xp.asarray(numpy.nanmax(X, axis=axis))


### 4. 데이터셋 분리

In [6]:
# LSTM 입력 형태로 데이터 변환
def create_lstm_data(df, lookback=60):
    X, y = [], []
    for i in range(len(df) - lookback):
        X.append(df.iloc[i:i + lookback, :-1].values)  # lookback 동안의 feature 값들
        y.append(df['Target'].iloc[i + lookback])      # 그 다음날 Target
    return np.array(X), np.array(y)

In [7]:
# 훈련 및 테스트 데이터 분할
train_data = data.loc['2021-01-01':'2023-12-31']
test_data = data.loc['2024-01-01':'2024-09-30']

# LSTM 입력을 위한 데이터 생성
X_train, y_train = create_lstm_data(train_data)
X_test, y_test = create_lstm_data(test_data)

### 5. XGBoost 모델

In [11]:
from sklearn.impute import SimpleImputer
from scikeras.wrappers import KerasClassifier
from sklearn.model_selection import GridSearchCV
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

In [20]:
X_train_2d = X_train.reshape(original_shape[0], -1)

# NaN 값 대체 (평균값을 사용하여 결측값 채움)
X_train_df = pd.DataFrame(X_train_2d)
X_train_df = X_train_df.fillna(X_train_df.mean())

# 결측값 대체 후 데이터 크기 확인
X_train_2d = X_train_df.to_numpy()
if X_train_2d.size == expected_size:
    # 크기가 일치하는 경우에만 3차원으로 변환
    X_train = X_train_2d.reshape(original_shape)
else:
    raise ValueError("데이터의 크기가 일치하지 않아 3차원으로 변환할 수 없습니다.")


In [25]:
# NaN 값 확인
print("초기 NaN 값 개수:", np.isnan(X_train).sum())

# 3차원 -> 2차원 변환 후 NaN 값 처리
if np.isnan(X_train).any():
    # 3차원 -> 2차원 변환
    X_train_2d = X_train.reshape(X_train.shape[0], -1)
    X_train_df = pd.DataFrame(X_train_2d)
    
    # 결측값을 각 열의 평균으로 채우고, NaN 제거 후 다시 확인
    X_train_df = X_train_df.fillna(X_train_df.mean())
    X_train_df = X_train_df.dropna()  # 혹시 남아 있을 수 있는 NaN 행 제거
    
    # NaN이 제거되었는지 확인
    print("결측값 채운 후 NaN 값 개수 (2차원):", X_train_df.isna().sum().sum())
    
    # 2차원 -> 3차원 변환
    X_train = X_train_df.to_numpy().reshape(X_train.shape[0], X_train.shape[1], X_train.shape[2])

# 최종 NaN 값 확인
print("NaN 값 제거 후 개수:", np.isnan(X_train).sum())

초기 NaN 값 개수: 41580
결측값 채운 후 NaN 값 개수 (2차원): 0


ValueError: cannot reshape array of size 0 into shape (693,60,9)

In [22]:
# 모델 생성 함수 정의
def create_model(optimizer='adam'):
    model = Sequential()
    model.add(Dense(64, input_shape=(X_train.shape[1],), activation='relu'))
    model.add(Dense(1, activation='sigmoid'))
    model.compile(optimizer=optimizer, loss='binary_crossentropy', metrics=['accuracy'])
    return model

# KerasClassifier에 모델 함수 전달
model = KerasClassifier(build_fn=create_model, verbose=0)

# 파라미터 그리드 설정
param_grid = {'optimizer': ['adam', 'rmsprop'], 'epochs': [10, 20]}

# GridSearchCV 설정
grid = GridSearchCV(estimator=model, param_grid=param_grid, scoring='accuracy', cv=3)

# Grid Search 수행
grid_result = grid.fit(X_train, y_train)

# 최적의 파라미터와 정확도 출력
print(f'최적의 파라미터: {grid_result.best_params_}')
print(f'최고의 정확도: {grid_result.best_score_}')

ValueError: 
All the 12 fits failed.
It is very likely that your model is misconfigured.
You can try to debug the error by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
12 fits failed with the following error:
Traceback (most recent call last):
  File "C:\Users\조영서\AppData\Roaming\Python\Python311\site-packages\sklearn\model_selection\_validation.py", line 888, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "C:\Users\조영서\AppData\Roaming\Python\Python311\site-packages\scikeras\wrappers.py", line 1501, in fit
    super().fit(X=X, y=y, sample_weight=sample_weight, **kwargs)
  File "C:\Users\조영서\AppData\Roaming\Python\Python311\site-packages\scikeras\wrappers.py", line 770, in fit
    self._fit(
  File "C:\Users\조영서\AppData\Roaming\Python\Python311\site-packages\scikeras\wrappers.py", line 925, in _fit
    X, y = self._initialize(X, y)
           ^^^^^^^^^

In [ ]:
# LSTM 모델 정의
model = Sequential([
    LSTM(64, return_sequences=True, input_shape=(X_train.shape[1], X_train.shape[2])),
    Dropout(0.2),
    LSTM(32),
    Dropout(0.2),
    Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# 조기 종료 설정
early_stopping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

# 모델 학습
history = model.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=50, batch_size=32, callbacks=[early_stopping])

### 6. 평가

In [ ]:
# 예측 및 평가
y_pred = (model.predict(X_test) > 0.5).astype(int)
accuracy = accuracy_score(y_test, y_pred)
print(f"LSTM 모델 정확도: {accuracy * 100:.2f}%")

In [ ]:
from sklearn.metrics import accuracy_score

# 초기 투자 금액 설정
initial_investment = 1000000  # 100만원
investment = initial_investment

# 테스트 데이터의 종가 가져오기
X_test_prices = test_data['Close'].iloc[-len(y_pred):].values

# 수익률 계산
for i in range(len(y_pred) - 1):
    if y_pred[i] == 1:  # 주가 상승 예측
        if y_test[i] == 1:  # 실제 주가 상승
            investment *= 1.01  # 1% 수익
        else:  # 실제 주가 하락
            investment *= 0.99  # 1% 손실
    elif y_pred[i] == 0:  # 주가 하락 예측
        if y_test[i] == 0:  # 실제 주가 하락
            investment *= 1.01  # 1% 수익
        else:  # 실제 주가 상승
            investment *= 0.99  # 1% 손실

# 최종 수익률 계산
final_profit = (investment - initial_investment) / initial_investment * 100
print(f"최종 투자 금액: {investment:.2f}원")
print(f"최종 수익률: {final_profit:.2f}%")


In [ ]:
print(f"Test data size: {len(test_data)}")
print(f"X_test shape: {X_test.shape}")
print(f"y_test shape: {y_test.shape}")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
import numpy as np

# 테스트 데이터를 사용하여 예측
y_pred = model.predict(X_test)

# NaN 값이 있는지 확인하고 처리
if np.isnan(y_pred).any():
    print("Warning: y_pred contains NaN values. They will be replaced with 0.")
    y_pred = np.nan_to_num(y_pred)  # NaN 값을 0으로 대체

# 예측 값을 0과 1로 변환
y_pred = (y_pred > 0.5).astype(int)

# 혼동 행렬 계산
cm = confusion_matrix(y_test, y_pred)

# 혼동 행렬 시각화
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Down', 'Up'], yticklabels=['Down', 'Up'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.show()

# 평가 지표 출력
print(classification_report(y_test, y_pred))
print(f'정확도: {accuracy_score(y_test, y_pred):.2f}')


### 7. 다른 주식에 적용

## XGBOOST

In [ ]:
import pandas as pd
import numpy as np
import yfinance as yf

### 1. 종목, 기간 설정

In [ ]:
# 엔비디아, TSMC, Broadcom, ASML, AMD
symbols = ["NVDA", "TSM", "AVGO", "ASML", "AMD"] 

all_data = []

start = "2023-01-01"
end = "2024-07-01"

### 2. 피처 엔지니어링 : MACD, 볼린저밴드, RSI

In [ ]:
# MACD 계산 함수
def add_macd(df, short_window=12, long_window=26, signal_window=9):
    df['EMA12'] = df['Close'].ewm(span=short_window, adjust=False).mean()
    df['EMA26'] = df['Close'].ewm(span=long_window, adjust=False).mean()
    df['MACD'] = df['EMA12'] - df['EMA26']
    df['Signal_Line'] = df['MACD'].ewm(span=signal_window, adjust=False).mean()
    df['MACD_Hist'] = df['MACD'] - df['Signal_Line']
    return df

# 볼린저 밴드 계산 함수
def add_bollinger_bands(df, window=20):
    df['MA20'] = df['Close'].rolling(window=window).mean()
    df['BB_upper'] = df['MA20'] + 2 * df['Close'].rolling(window=window).std()
    df['BB_lower'] = df['MA20'] - 2 * df['Close'].rolling(window=window).std()
    df['BB_width'] = df['BB_upper'] - df['BB_lower']
    return df

# RSI 계산 함수
def add_rsi(df, window=14):
    delta = df['Close'].diff(1)
    gain = np.where(delta > 0, delta, 0)
    loss = np.where(delta < 0, -delta, 0)

    avg_gain = pd.Series(gain).rolling(window=window, min_periods=1).mean()
    avg_loss = pd.Series(loss).rolling(window=window, min_periods=1).mean()
    rs = avg_gain / avg_loss
    df['RSI'] = np.where(avg_loss == 0, 100, 100 - (100 / (1 + rs)))
    return df

### 3. 데이터 전처리

In [ ]:
from sklearn.preprocessing import MinMaxScaler

for symbol in symbols:
    df = yf.download(symbol, start, end)

    df = add_macd(df)
    df = add_bollinger_bands(df)
    df = add_rsi(df)
    
    # 타겟 생성 (다음 날의 주가 상승 여부)
    df['Target'] = (df['Close'].shift(-1) > df['Close']).astype(int)
    
    df = df.fillna(method='ffill').dropna()
    
    # 날짜 인덱스가 시간 순서대로 정렬되었는지 확인 및 정렬
    df = df.sort_index()

    # columns 정의
    scale_cols = ['Open', 'High', 'Low', 'Close', 'Volume', 'MACD', 'Signal_Line', 'MACD_Hist', 'MA20', 'BB_upper', 'BB_lower', 'BB_width', 'RSI']
    
    # 스케일링
    scaler = MinMaxScaler()
    scaled = scaler.fit_transform(df[scale_cols])
    scaled_df = pd.DataFrame(scaled, columns=scale_cols, index=df.index)
    
    # 스케일된 데이터와 타겟 결합
    scaled_df['Target'] = df['Target'].values
    all_data.append(scaled_df)

data = pd.concat(all_data, axis=0, ignore_index=False)

# 인덱스가 datetime 형식인지 확인 및 변환
data.index = pd.to_datetime(data.index)
# 날짜가 시간 순서대로 정렬되어 있는지 확인
data = data.sort_index()


### 4. 데이터셋 분리

In [ ]:
# 시간 기반으로 훈련 및 테스트 데이터 분할
train_data = data.loc['2023-01-01':'2024-12-31']  # 훈련 데이터
test_data = data.loc['2024-01-01':'2024-07-01']   # 테스트 데이터

# 독립 변수(X)와 종속 변수(y) 설정
X_train = train_data.drop(columns=['Target'])
y_train = train_data['Target']
X_test = test_data.drop(columns=['Target'])
y_test = test_data['Target']

### 5. XGBoost 모델

In [ ]:
from sklearn.model_selection import GridSearchCV, KFold
from xgboost import XGBClassifier

# 그리드 서치에 사용할 하이퍼파라미터 그리드 정의
param_grid = {
    'n_estimators': [35, 40, 45],
    'max_depth': [3, 4, 5],
    'learning_rate': [0.008, 0.01, 0.02],
    'subsample': [0.9, 0.95],
    'colsample_bytree': [0.85, 0.9, 0.95],
    'gamma': [0.25, 0.3, 0.35],
    'min_child_weight': [1]
}

xgb = XGBClassifier(eval_metric='logloss')

xgboost = GridSearchCV(
    estimator=xgb,
    param_grid=param_grid,
    scoring='accuracy',  # 모델 성능을 평가할 지표 (필요에 따라 변경 가능)
    cv=KFold(n_splits = 3),  # 교차 검증 폴드 수
    n_jobs=-1  # 병렬 처리 사용 (모든 프로세서 사용)
)
# 그리드 서치 실행 (훈련 데이터를 사용)
xgboost.fit(X_train, y_train)

### 6. 평가

In [ ]:
# 최적의 하이퍼파라미터 출력
print("Best parameters found: ", xgboost.best_params_)
print("Best Accuracy: {:.1f}%".format(xgboost.best_score_ * 100))

In [ ]:
y_pred = xgboost.predict(X_test)

# 초기 투자금 설정
initial_investment = 1000000  # 100만 원

investment = initial_investment
holding = False  # 현재 주식을 보유 중인지 여부
num_shares = 0   # 구매한 주식 수

X_test['Close'] = test_data['Close']

# 각 날의 수익률을 계산
for i in range(len(y_test) - 1):
    if y_pred[i] == 1 and not holding:  # 상승 예측 & 주식을 보유하지 않은 경우 매수
        # 전날 종가로 매수
        num_shares = investment / X_test['Close'].iloc[i]
        holding = True
    
    elif y_pred[i] == 0 and holding:  # 하락 예측 & 주식을 보유한 경우 매도
        # 오늘 종가로 매도
        investment = num_shares * X_test['Close'].iloc[i]
        holding = False
        num_shares = 0

if holding:
    investment = num_shares * X_test['Close'].iloc[-1]

# 최종 수익률 계산
final_profit = (investment - initial_investment) / initial_investment * 100

print(f"최종 투자 금액: {investment:.2f}원")
print(f"최종 수익률: {final_profit:.2f}%")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score

# 테스트 데이터를 사용하여 예측
y_pred = xgboost.predict(X_test)

# 혼동 행렬 계산
cm = confusion_matrix(y_test, y_pred)

# 혼동 행렬 시각화
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Down', 'Up'], yticklabels=['Down', 'Up'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.show()

In [ ]:
# 상관계수 행렬
correlation_matrix = data.corr()
top_3_features = correlation_matrix['Target'].sort_values(ascending=False).index[1:4]
top_3_features

In [ ]:
# 피처 중요도 추출
importances = xgboost.best_estimator_.feature_importances_

# 피처 중요도를 시각화
feature_names = X_train.columns
importance_df = pd.DataFrame({'Feature': feature_names, 'Importance': importances})
importance_df = importance_df.sort_values(by='Importance', ascending=False)

sns.barplot(x='Importance', y='Feature', data=importance_df)
plt.title('Feature Importance')
plt.xlabel('Importance')
plt.ylabel('Features')
plt.xlim(0.03, 0.13)
plt.show()


### 7. 다른 주식에 적용

In [ ]:
# 테스트할 종목 리스트
test_symbols = ['KLAC', 'MRVL', 'MPWR', 'MCHP', 'ON']

# 정확도를 저장할 딕셔너리
accuracy_dict = {}

for symbol in test_symbols:

    test_data = yf.download(symbol, start=start, end=end)
    
    test_data = add_macd(test_data)
    test_data = add_bollinger_bands(test_data)
    test_data = add_rsi(test_data)
    
    test_data['Target'] = (test_data['Close'].shift(-1) > test_data['Close']).astype(int)
    
    test_data = test_data.fillna(method='ffill').dropna()
    test_data = test_data.sort_index()
    
    scaled = scaler.transform(test_data[scale_cols])
    scaled_df = pd.DataFrame(scaled, columns=scale_cols, index=test_data.index)
    scaled_df['Target'] = test_data['Target'].values
    
    X_test_symbol = scaled_df.drop(columns=['Target'])
    y_test_symbol = scaled_df['Target']
    
    y_pred_symbol = xgboost.predict(X_test_symbol)
    
    accuracy = accuracy_score(y_test_symbol, y_pred_symbol)
    accuracy_dict[symbol] = accuracy

# 정확도 딕셔너리를 DataFrame으로 변환
accuracy_df = pd.DataFrame(list(accuracy_dict.items()), columns=['Symbol', 'Accuracy'])

In [ ]:
accuracy_df

In [ ]:
sns.barplot(x='Symbol', y='Accuracy', data=accuracy_df)
plt.title('Prediction Accuracy for Different Stocks')
plt.xlabel('Stock Symbol')
plt.ylabel('Accuracy')
plt.ylim(0.5, 0.55)
plt.show()